# Exoplanet Taxonomic Clustering
**Goal:** Unsupervised clustering of exoplanets into physical classes (terrestrial, sub-Neptune, gas giant) using radius, mass, period, semi-major axis, eccentricity, and temperature.


**Pipeline:** Audit $\rightarrow$ EDA $\rightarrow$ Log Preprocessing $\rightarrow$ K-Means & Hierarchical Clustering $\rightarrow$ Evaluation.


## 1. Setup & Environment
Initializes random seeds (`seed=42`) and colorblind-safe styling for reproducible clustering and accessible visualizations.


In [2]:
# Core system and data libraries
import sys
import os
import warnings
import time

# Numerical and tabular processing
import numpy as np
import pandas as pd

# Data visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Hierarchical clustering and dendrogram generation
from scipy.cluster.hierarchy import linkage, dendrogram

# Preprocessing, clustering algorithms, and evaluation metrics
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Global environment settings
warnings.filterwarnings('ignore')
np.random.seed(42)

# Visualization aesthetics: Colorblind-friendly palette and publication styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
sns.set_palette('colorblind')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#333333'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['figure.dpi'] = 120

print("Environment Initialized Successfully:")
print(f" - Python Version     : {sys.version.split()[0]}")
print(f" - NumPy Version      : {np.__version__}")
print(f" - Pandas Version     : {pd.__version__}")
import sklearn
print(f" - Scikit-Learn Version: {sklearn.__version__}")


Environment Initialized Successfully:
 - Python Version     : 3.12.0
 - NumPy Version      : 2.5.0
 - Pandas Version     : 3.0.5
 - Scikit-Learn Version: 1.9.0


## 2. Data Ingestion & Audit
Loaded 6,445 confirmed exoplanets across 6 physical features, dropping non-physical discovery metadata and missing stellar columns.
Eccentricity has the highest missing rate (16.5%), while radius and mass are under 1% missing.

In [3]:
# Define file path
dataset_path = 'D:\Desktop\exoplanets.csv'

# Complete candidate feature schema requested
target_requested_columns = [
    'pl_name',     # Planet Name (identifier)
    'pl_rade',     # Planet Radius [Earth radii]
    'pl_bmasse',   # Planet Mass [Earth masses]
    'pl_orbper',   # Orbital Period [days]
    'pl_orbsmax',  # Orbital Semimajor Axis [AU]
    'pl_orbeccen', # Orbital Eccentricity
    'pl_eqt',      # Equilibrium Temperature [K]
    'st_teff',     # Host Star Effective Temperature [K]
    'st_mass',     # Host Star Mass [Solar masses]
    'st_rad'       # Host Star Radius [Solar radii]
]

# Load raw dataset
raw_df = pd.read_csv(dataset_path)
print(f"Raw Dataset Ingested: {raw_df.shape[0]} rows, {raw_df.shape[1]} columns.")
print(f"Raw Columns: {list(raw_df.columns)}")

# Audit schema against available columns
available_target_cols = [c for c in target_requested_columns if c in raw_df.columns]
missing_target_cols = [c for c in target_requested_columns if c not in raw_df.columns]
extraneous_cols = [c for c in raw_df.columns if c not in target_requested_columns]

print(f"\nAvailable Target Columns ({len(available_target_cols)}): {available_target_cols}")
print(f"Requested but Absent Columns ({len(missing_target_cols)}): {missing_target_cols}")
print(f"Extraneous Metadata Columns Dropped ({len(extraneous_cols)}): {extraneous_cols}")

# Filter dataset to available target columns
df_filtered = raw_df[available_target_cols].copy()

# Set identifier 'pl_name' as dataframe index to isolate it from feature matrices
if 'pl_name' in df_filtered.columns:
    df_filtered.set_index('pl_name', inplace=True)

# Separate numerical feature matrix
feature_names = list(df_filtered.columns)
print(f"\nActive Feature Matrix Columns ({len(feature_names)}): {feature_names}")

# Structural Audit: Info and Head
print("\n--- Feature Data Types & Non-Null Counts ---")
df_filtered.info()

print("\n--- First 5 Exoplanet Records in Pipeline ---")
display(df_filtered.head())

# Missing value audit
missing_audit = pd.DataFrame({
    'Missing_Count': df_filtered.isnull().sum(),
    'Missing_Percentage': (df_filtered.isnull().sum() / len(df_filtered)) * 100
}).sort_values(by='Missing_Count', ascending=False)

print("\n--- Missing Value Profile ---")
display(missing_audit)


Raw Dataset Ingested: 6445 rows, 8 columns.
Raw Columns: ['pl_name', 'pl_bmasse', 'pl_rade', 'pl_orbper', 'pl_orbeccen', 'pl_orbsmax', 'pl_eqt', 'disc_year']

Available Target Columns (7): ['pl_name', 'pl_rade', 'pl_bmasse', 'pl_orbper', 'pl_orbsmax', 'pl_orbeccen', 'pl_eqt']
Requested but Absent Columns (3): ['st_teff', 'st_mass', 'st_rad']
Extraneous Metadata Columns Dropped (1): ['disc_year']

Active Feature Matrix Columns (6): ['pl_rade', 'pl_bmasse', 'pl_orbper', 'pl_orbsmax', 'pl_orbeccen', 'pl_eqt']

--- Feature Data Types & Non-Null Counts ---
<class 'pandas.DataFrame'>
Index: 6445 entries, HD 2039 b to TOI-789 c
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   pl_rade      6395 non-null   float64
 1   pl_bmasse    6414 non-null   float64
 2   pl_orbper    6092 non-null   float64
 3   pl_orbsmax   5953 non-null   float64
 4   pl_orbeccen  5379 non-null   float64
 5   pl_eqt       5909 non-null   float64
dty

,pl_rade,pl_bmasse,pl_orbper,pl_orbsmax,pl_orbeccen,pl_eqt
pl_name,,,,,,
HD 2039 b,12.700000,1999.1507,1120.000000,2.20000,0.71,210.84
HAT-P-8 b,15.692600,406.8224,3.076340,0.04496,0.00,1713.00
K2-43 b,4.510000,18.5000,3.471149,0.03784,0.07,939.30
Kepler-1753 b,2.226781,5.5900,16.004601,0.11790,0.00,583.00
Kepler-1176 b,2.450000,6.5700,24.173858,0.15890,0.00,595.00



--- Missing Value Profile ---


,Missing_Count,Missing_Percentage
pl_orbeccen,1066,16.539953
pl_eqt,536,8.316524
pl_orbsmax,492,7.633825
pl_orbper,353,5.477114
pl_rade,50,0.775795
pl_bmasse,31,0.480993
